# Some experiments using "Seismic" data

The data used below for the experiments with decision reducts and bireducts is derived from the original dataset used in a data mining challenge hosted on the Knowledgepit platform https://knowledgepit.ml.

For the summary of the challenge please refer to:

```
Andrzej Janusz, Dominik Ślęzak, Marek Sikora, Łukasz Wróbel:
Predicting Dangerous Seismic Events: AAIA'16 Data Mining Challenge. FedCSIS 2016: 205-211
```

or to the data mining challenge page, where the description, results, as well as, the original data can be found - https://knowledgepit.ml/aaia16-data-mining-challenge.

---

We applied feature extraction and unsupervised discretization (quantile-based) to create a dataset for the experiments below - these operations are not included in the notebook, just the final dataset (cf. ``data`` directory). Please note that we only used the training data from the data mining challenge as our dataset for the experiments, splitting it into train-test subsets. It's important to mention that this approach deviates from best practices in machine learning because a random split of a dataset involving or originating from time-series data may introduce data leaks. However, for the purpose of this notebook, which focuses on demonstrating the utilization of decision reducts and bireducts, we will disregard this fact.


In [ ]:
# Install the required packages below

# !pip install more-itertools==8.14.0
# !pip install numpy==1.25.2
# !pip install pandas==2.1.0
# !pip install scikit-learn==1.3.1
# !pip install scikit-rough==0.2.0
# !pip install snakeviz==2.2.0
# !pip install xgboost==2.0.0

In [ ]:
import functools
import pathlib
from collections import defaultdict

import more_itertools
import numpy as np
import pandas as pd
from sklearn.compose import make_column_transformer
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier
from skrough.algorithms.reducts import (
    get_approx_reduct_greedy_heuristic,
    get_approx_reduct_daar_heuristic,
)
from skrough.dataprep import prepare_factorized_array, prepare_factorized_vector
from skrough.disorder_measures import gini_impurity
from skrough.feature_importance import (
    get_feature_importance,
)
from skrough.predict.predict_attrs_ensemble import predict_attrs_ensemble
from skrough.predict.predict_objs_attrs_ensemble import predict_objs_attrs_ensemble
from xgboost import XGBClassifier

rng = np.random.default_rng()

# data preparation

In [2]:
DATA_DIR = pathlib.Path("data")

data = pd.read_csv(DATA_DIR / "seismic_training_data_features_discretized.csv")
target = pd.read_csv(DATA_DIR / "seismic_training_labels.csv", header=None)[0]

# drop main_working_id column
data.drop("main_working_id", axis=1, inplace=True)
column_names = data.columns.to_list()

# encode data and target into the form of integers
data = OrdinalEncoder().fit_transform(data)
target = LabelEncoder().fit_transform(target)

# split into train and test subsets
train, test, train_target, test_target = train_test_split(
    data, target, test_size=0.10, random_state=42
)

# Computation of greedy and DAAR-based reducts

Let's use a greedy approach and a function-based DAAR (Dynamically Adjusted Approximation Reducts) heuristic approach:

```
Andrzej Janusz, Dominik Ślęzak:
Computation of Approximate Reducts with Dynamically Adjusted Approximation Threshold. ISMIS 2015: 19-28
```

In [ ]:
greedy_heuristic_reducts = get_approx_reduct_greedy_heuristic(
    x=train,
    y=train_target,
    disorder_fun=gini_impurity,
    epsilon=0.05,
    candidates_count=100,
    n_reducts=100,
    n_jobs=-1,
)

In [ ]:
greedy_heuristic_reducts

[AttrsSubset(attrs=[201, 167, 15, 8, 313, 294, 516, 207, 6, 338, 601, 494, 493, 603, 75, 5, 538]),
 AttrsSubset(attrs=[125, 0, 6, 203, 4, 605, 400, 334, 471, 316, 123, 517, 493, 54, 312, 190, 121]),
 AttrsSubset(attrs=[3, 8, 653, 15, 123, 142, 407, 6, 516, 489, 334, 490, 5, 54, 341, 317, 185]),
 AttrsSubset(attrs=[6, 386, 142, 168, 189, 494, 4, 491, 5, 400, 98, 517, 55, 512, 338, 597]),
 AttrsSubset(attrs=[6, 593, 142, 229, 401, 313, 471, 4, 514, 123, 338, 494, 74, 510, 335, 537]),
 AttrsSubset(attrs=[8, 123, 461, 4, 208, 6, 5, 511, 318, 399, 334, 76, 491, 187, 397, 75]),
 AttrsSubset(attrs=[8, 123, 593, 407, 296, 190, 4, 315, 5, 514, 338, 736, 6, 54, 185, 121]),
 AttrsSubset(attrs=[167, 191, 392, 168, 4, 7, 6, 403, 513, 334, 98, 490, 516, 493, 77, 185]),
 AttrsSubset(attrs=[190, 123, 275, 4, 317, 6, 492, 315, 294, 336, 185, 517, 76, 7, 53, 403]),
 AttrsSubset(attrs=[168, 123, 143, 4, 6, 595, 515, 489, 492, 516, 334, 119, 539, 76, 402]),
 AttrsSubset(attrs=[189, 124, 11, 142, 511, 295,

In [ ]:
daar_heuristic_reducts = get_approx_reduct_daar_heuristic(
    x=train,
    y=train_target,
    disorder_fun=gini_impurity,
    candidates_count=100,
    allowed_randomness=0.05,
    probes_count=100,
    fast=True,
    n_reducts=100,
    n_jobs=-1,
)

In [ ]:
daar_heuristic_reducts

[AttrsSubset(attrs=[123, 124, 6, 291, 296, 488, 5, 493, 339, 4, 514]),
 AttrsSubset(attrs=[0, 8, 650, 395, 404, 124, 511, 411, 604, 4, 399, 492, 338]),
 AttrsSubset(attrs=[8, 189, 534, 7, 6, 729, 208, 515, 492, 491]),
 AttrsSubset(attrs=[8, 652, 0, 11, 167, 6, 736, 250, 731, 513, 340, 4, 318]),
 AttrsSubset(attrs=[168, 8, 6, 452, 470, 207, 5, 732, 515, 123, 510]),
 AttrsSubset(attrs=[15, 412, 189, 190, 8, 4, 488, 7, 313, 143, 340, 463]),
 AttrsSubset(attrs=[123, 8, 659, 7, 124, 464, 15, 511, 726, 6, 406, 5, 4]),
 AttrsSubset(attrs=[680, 6, 725, 404, 4, 295, 335, 252, 319, 412, 488, 5]),
 AttrsSubset(attrs=[6, 533, 470, 190, 412, 597, 296, 515, 490, 5]),
 AttrsSubset(attrs=[123, 458, 125, 8, 124, 470, 315, 4, 207, 514, 515, 732, 337]),
 AttrsSubset(attrs=[0, 15, 458, 11, 8, 677, 294, 5, 7, 737, 603, 319, 513, 339]),
 AttrsSubset(attrs=[286, 168, 8, 458, 189, 6, 274, 318, 5, 466, 465, 488]),
 AttrsSubset(attrs=[189, 8, 7, 395, 295, 208, 407, 600, 318, 513, 337]),
 AttrsSubset(attrs=[6, 4

# Show stats for computed reducts

Let us present some statistics of the computed sets of reducts, e.g,. avg, min, max lenght of computed reducts, or the most frequently used columns/attributes.

In [10]:
def stats(reds, name, top, for_objs=False):
    attr_counts = defaultdict(int)
    attrs_lengths = []
    objs_lengths = []

    for elem in reds:
        attrs_lengths.append(len(elem.attrs))
        if for_objs:
            objs_lengths.append(len(elem.objs))
        for a in elem.attrs:
            attr_counts[a] += 1

    attr_counts_df = pd.DataFrame(attr_counts.items(), columns=["attr", "count"])
    attr_counts_df = attr_counts_df.set_index("attr")

    print(f"no of {name}s        = {len(reds)}")
    print(f"avg {name} length    = {np.mean(attrs_lengths)}")
    print(f"median {name} length = {np.median(attrs_lengths)}")
    print(f"min {name} length    = {np.min(attrs_lengths)}")
    print(f"max {name} length    = {np.max(attrs_lengths)}")
    if for_objs:
        print(f"avg {name} objs    = {np.mean(objs_lengths)}")
        print(f"median {name} objs = {np.median(objs_lengths)}")
        print(f"min {name} objs    = {np.min(objs_lengths)}")
        print(f"max {name} objs    = {np.max(objs_lengths)}")
    print("most commonly used attrs (attrs positions starting from 0):")
    print(attr_counts_df.sort_values("count", ascending=False)[:top])


def stats_reducts(reducts, top=10):
    stats(reducts, name="reduct", top=top, for_objs=False)

In [ ]:
stats_reducts(greedy_heuristic_reducts)

no of reducts        = 100
avg reduct length    = 16.38
median reduct length = 16.0
min reduct length    = 15
max reduct length    = 18
most commonly used attrs (attrs positions starting from 0):
      count
attr       
6        93
4        63
5        51
123      39
124      32
512      30
189      28
190      27
8        26
492      26


In [ ]:
stats_reducts(daar_heuristic_reducts)

no of reducts        = 100
avg reduct length    = 11.89
median reduct length = 12.0
min reduct length    = 10
max reduct length    = 14
most commonly used attrs (attrs positions starting from 0):
      count
attr       
6        83
8        55
4        52
5        42
190      32
189      31
123      31
15       29
168      28
167      24


# Performance of classifiers

The dataset is imbalanced - there is a lot more entities with the one decision than other. So, for performance we will be mainly interested in BAC (Balanced Accuracy) but for reference we also present ACC (Accuracy).

In [ ]:
# see the distribution of the target attribute
pd.Series(target).value_counts()

0    130188
1      2963
Name: count, dtype: int64

In [14]:
# helper functions to present performance measures


def print_perf_values(bac, acc):
    print(f"BAC: {bac}")
    print(f"ACC: {acc}")


def performance_cl(cl):
    """Print performance for a sklearn-compatible classifier."""
    print_perf_values(
        bac=balanced_accuracy_score(test_target, cl.predict(test)),
        acc=accuracy_score(test_target, cl.predict(test)),
    )


def performance_reducts(reducts):
    """Print performance for a set of reducts - they are used as an ensemle of rule-based classifiers."""
    predictions = predict_attrs_ensemble(
        model_ensemble=reducts,
        reference_data=train,
        reference_data_y=train_target,
        predict_data=test,
        fill_missing=0,
    )
    print_perf_values(
        bac=balanced_accuracy_score(test_target, predictions),
        acc=accuracy_score(test_target, predictions),
    )


def performance_bireducts(bireducts):
    """Print performance for a set of bireducts - they are used as an ensemle of rule-based classifiers."""
    predictions = predict_objs_attrs_ensemble(
        model_ensemble=bireducts,
        reference_data=train,
        reference_data_y=train_target,
        predict_data=test,
        fill_missing=0,
    )
    print_perf_values(
        bac=balanced_accuracy_score(test_target, predictions),
        acc=accuracy_score(test_target, predictions),
    )

In [15]:
# performance for Decition Tree Classifier (default settings)

cl = DecisionTreeClassifier()
cl.fit(train, train_target)
performance_cl(cl)

BAC: 0.8338824501595168
ACC: 0.9851306698708321


In [16]:
# performance for Random Forest Classifier (100 estimators + default settings)

cl = RandomForestClassifier(n_estimators=100)
cl.fit(train, train_target)
performance_cl(cl)

BAC: 0.7613405313029951
ACC: 0.9879092820666867


In [17]:
# performance for XGBoost (100 estimators + default settings)

cl = XGBClassifier(n_estimators=100, objective="binary:logistic")
cl.fit(train, train_target)
performance_cl(cl)

BAC: 0.8241538251485261
ACC: 0.9897116251126464


In [18]:
# performance for greedy heuristic reducts

performance_reducts(greedy_heuristic_reducts)

BAC: 0.8440467705997442
ACC: 0.9902373085010514


In [ ]:
# performance for daar heuristic reducts

performance_reducts(daar_heuristic_reducts)

BAC: 0.7270539101907457
ACC: 0.985806548513067


# feature selection

As we could see above, the performance computed for reducts and bireduct (used as rule-based classifiers) is lower than for, e.g., Decision Tree or XGBoost. In some cases they can perform well, but quite often such simple rule-based classifiers are just worse than other more advanced methods. 

The main strength of reducts and bireducts may not be in using them directly as classifiers (as seen above) but as a feature selection method that can help more advanced classifiers perform even better.


In [20]:
def get_attrs_union(reducts):
    """Get union of attribute numbers that are used in the given ``reducts``."""
    return list(functools.reduce(lambda s, red: s | set(red.attrs), reducts, set()))


# e.g., attr numbers used in reducts[0] and reducts[1]
print(get_attrs_union(greedy_heuristic_reducts[:2]))

# e.g., attr numbers used in reducts[1], reducts[2], reducts[3], reducts[4]
print(get_attrs_union(greedy_heuristic_reducts[1:5]))

[0, 516, 5, 6, 4, 8, 201, 517, 75, 203, 334, 15, 207, 400, 338, 471, 601, 538, 603, 605, 121, 294, 167, 493, 494, 54, 312, 313, 123, 316, 125, 190]
[0, 512, 386, 3, 4, 517, 6, 516, 5, 8, 514, 653, 142, 15, 400, 401, 407, 537, 168, 54, 55, 312, 185, 313, 316, 317, 190, 189, 74, 203, 334, 335, 593, 338, 341, 597, 471, 605, 98, 229, 489, 490, 491, 493, 494, 121, 123, 125, 510]


In [ ]:
def get_attrs_merged_in_n(reducts, n: int):
    """Utilize all provided ``reducts`` to randomly shuffle and pair them into tuples
    (pairs, triplets, quintets, etc., depending on ``n``), then combine the attr numbers
    within the resulting tuples.

    When the given number of reducts cannot be split evenly into sets that have exactly ``n`` elements
    then some of them will be smaller.
    """
    shuffled = rng.choice(reducts, size=len(reducts), replace=False)
    return list(map(get_attrs_union, more_itertools.chunked_even(shuffled, n=n)))


# e.g., attrs used in all reducts
attrs_all = get_attrs_union(greedy_heuristic_reducts)

# e.g., randomly shuffle and pair reducts in triplets, join attr numbers within triplets
attrs_in_triplets = get_attrs_merged_in_n(greedy_heuristic_reducts, n=3)

# e.g., randomly shuffle and pair reducts in quintets, join attr numbers within quintets
attrs_in_quintets = get_attrs_merged_in_n(greedy_heuristic_reducts, n=5)

In [ ]:
# Let's prepare an sklearn Pipeline that use only attrs (attrs_all) used in reducts.
# Then, fit the classifier and print its performance.

cl = Pipeline(
    [
        (
            "feature_selection",
            make_column_transformer(("passthrough", attrs_all)),
        ),
        ("cl", DecisionTreeClassifier()),
    ]
)

cl.fit(train, train_target)

performance_cl(cl)

BAC: 0.8225077852113272
ACC: 0.9835536197056173


In [23]:
# A more complex example. Let's prepare an ensemble of Decision Tree Classifiers.
# Each component of the ensemble will use a subset of attr numbers that were used in a given triplet of reducts.

cl = VotingClassifier(
    [
        (
            f"cl{i}",
            Pipeline(
                [
                    (
                        "feature_selection",
                        make_column_transformer(("passthrough", attrs)),
                    ),
                    ("cl", DecisionTreeClassifier()),
                ]
            ),
        )
        for i, attrs in enumerate(attrs_in_triplets)
    ]
)
cl.fit(train, train_target)

performance_cl(cl)

BAC: 0.8582865172220622
ACC: 0.9915139681586062


# feature importance

Moreover, we can use functions from the library to assess attributes. Currently, the interface is not yet in its final shape, i.e., it is still a little bit low-level - there is need to prepare inputs used in the below functions (cf., ``prepare_factorized_array``, ``prepare_factorized_vector``). In future these operations will be hidden inside the functions.

In [30]:
x, x_counts = prepare_factorized_array(train)
y, y_count = prepare_factorized_vector(train_target)
feat_importance_greedy_heuristic = get_feature_importance(
    x=x,
    x_counts=x_counts,
    y=y,
    y_count=y_count,
    column_names=column_names,
    attrs_subsets=greedy_heuristic_reducts,
    disorder_fun=gini_impurity,
)

In [31]:
feat_importance_greedy_heuristic

,column,count,global_gain,avg_global_gain
0,total_bumps_energy,1.0,0.000324,0.000324
1,total_tremors_energy,0.0,0.000000,0.000000
2,total_destressing_blasts_energy,0.0,0.000000,0.000000
3,total_seismic_energy,2.0,0.000666,0.000333
4,latest_progress_estimation_l,63.0,0.081092,0.001287
...,...,...,...,...
733,max8_avg_difference_in_gactivity_W2,0.0,0.000000,0.000000
734,max8_avg_difference_in_gactivity_W3,2.0,0.001622,0.000811
735,max8_avg_difference_in_genergy_W1,2.0,0.002007,0.001003
736,max8_avg_difference_in_genergy_W2,9.0,0.009899,0.001100


In [32]:
# Let's sort the attributes according to the "count" column - these particular
# ordering should match the order we already have seen above in "Show stats for computed reducts and bireducts"
feat_importance_greedy_heuristic.sort_values("count", ascending=False)[
    ["column", "count", "global_gain"]
].iloc[:10]

,column,count,global_gain
6,latest_seismic_assessment,93.0,0.317649
4,latest_progress_estimation_l,63.0,0.081092
5,latest_progress_estimation_r,51.0,0.058277
123,max24_count_e3,39.0,0.041546
124,max24_count_e4,32.0,0.029762
512,whichminDiff24_avg_gactivity,30.0,0.034276
189,maxminDiff24_count_e3,28.0,0.030646
190,maxminDiff24_count_e4,27.0,0.025482
8,latest_comprehensive_assessment,26.0,0.021823
492,whichmaxDiff24_max_difference_in_gactivity,26.0,0.029856


In [33]:
# Let's see the attribute ordering when "global_gain" feature importance measure is used
feat_importance_greedy_heuristic.sort_values("global_gain", ascending=False)[
    ["column", "count", "global_gain"]
].iloc[:10]

,column,count,global_gain
6,latest_seismic_assessment,93.0,0.317649
4,latest_progress_estimation_l,63.0,0.081092
5,latest_progress_estimation_r,51.0,0.058277
123,max24_count_e3,39.0,0.041546
512,whichminDiff24_avg_gactivity,30.0,0.034276
189,maxminDiff24_count_e3,28.0,0.030646
492,whichmaxDiff24_max_difference_in_gactivity,26.0,0.029856
124,max24_count_e4,32.0,0.029762
334,whichmin24_max_gactivity,24.0,0.029226
121,lastminDiff24_avg_difference_in_genergy,24.0,0.027475


In [35]:
# Let's just use the attribute ranking to select small number of "most important" attributes
# with respect to the "count" measure. Get 30 best attrs and check Decision Tree Classifier
# performance on them.

best_cols = feat_importance_greedy_heuristic.sort_values(
    "count", ascending=False
).index[:30]

cl = Pipeline(
    [
        (
            "feature_selection",
            make_column_transformer(("passthrough", best_cols)),
        ),
        ("cl", DecisionTreeClassifier()),
    ]
)

cl.fit(train, train_target)

performance_cl(cl)

BAC: 0.8418619790270494
ACC: 0.9830279363172124
